<div dir="rtl">

# مشروع الترم ٢: موديل يتعرف على الأشكال

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-2/07-image-recognizer-project.ipynb)

هنعمل **dataset بنفسنا**: صور صغيرة لـ دايرة ومربع ومثلث، بأحجام وأماكن مختلفة وشوية "تشويش". وبعدين ندرّب موديلات تتعرف عليها ونقارن بينهم.

**دي نفس دورة حياة أي مشروع ذكاء اصطناعي:** بيانات ← تجهيز ← تدريب ← تقييم ← تحسين.

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SIZE = 20
CLASSES = ["circle", "square", "triangle"]
rng = np.random.default_rng(0)

<div dir="rtl">

## ١. نولّد الصور

</div>

In [ ]:
def draw_shape(kind: str, rng: np.random.Generator, size: int = SIZE, noise: float = 0.15) -> np.ndarray:
    yy, xx = np.mgrid[0:size, 0:size]
    r = rng.uniform(size * 0.2, size * 0.38)
    cx, cy = rng.uniform(r, size - r, 2)
    if kind == "circle":
        img = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
    elif kind == "square":
        img = (abs(xx - cx) <= r * 0.85) & (abs(yy - cy) <= r * 0.85)
    elif kind == "triangle":
        top, bottom = cy - r, cy + r
        half_width = (yy - top) / (2 * r) * r
        img = (yy >= top) & (yy <= bottom) & (abs(xx - cx) <= half_width)
    else:
        raise ValueError(kind)
    img = img.astype(float) + rng.normal(0, noise, (size, size))
    return img.clip(0, 1)


def make_dataset(n_per_class: int, rng: np.random.Generator, noise: float = 0.15):
    X, y = [], []
    for label, kind in enumerate(CLASSES):
        for _ in range(n_per_class):
            X.append(draw_shape(kind, rng, noise=noise).ravel())
            y.append(label)
    return np.array(X), np.array(y)


X, y = make_dataset(300, rng)
print("Dataset:", X.shape, "labels:", np.bincount(y))

fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for row, label in enumerate(range(3)):
    for ax, idx in zip(axes[row], np.where(y == label)[0][:8]):
        ax.imshow(X[idx].reshape(SIZE, SIZE), cmap="gray_r")
        ax.axis("off")
    axes[row][0].set_title(CLASSES[label], loc="left")
plt.show()

<div dir="rtl">

## ٢. ندرّب ٣ موديلات ونقارن

</div>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=0),
    "Neural Net": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400, random_state=0),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = model.score(X_test, y_test)
    print(f"{name:<14} {results[name]:.1%}")

plt.figure(figsize=(6, 3))
plt.bar(results.keys(), results.values(), color=["#4F46E5", "#D97706", "#059669"])
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.show()

<div dir="rtl">

## ٣. نحلل أحسن موديل

</div>

In [ ]:
best_name = max(results, key=results.get)
best = models[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=CLASSES))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, display_labels=CLASSES, cmap="Blues")
plt.show()

<div dir="rtl">

## ٤. اختبار الصمود: صور فيها تشويش أكتر

الموديل اتدرب على تشويش ٠.١٥. هل هيصمد لو الصور بقت أوحش؟ (زي كاميرا موبايل في إضاءة وحشة)

</div>

In [ ]:
for noise in [0.15, 0.3, 0.45, 0.6]:
    X_noisy, y_noisy = make_dataset(100, np.random.default_rng(1), noise=noise)
    accs = {name: m.score(X_noisy, y_noisy) for name, m in models.items()}
    print(f"noise={noise:.2f}  " + "  ".join(f"{k}={v:.0%}" for k, v in accs.items()))

<div dir="rtl">

**سؤال للنقاش:** لو عايزين الموديل يصمد في التشويش، نعمل إيه؟ (تلميح: افتكروا الحصة ٦ في مسار الأطفال عن البيانات المتنوعة.)

</div>

<div dir="rtl">

## 🏋️ مهام المشروع

1. **حسّن الصمود:** درّب الموديل على خليط من مستويات تشويش مختلفة، وقارن النتيجة في اختبار الصمود.
2. **شكل رابع:** ضيف شكل جديد (مثلاً `cross` ➕) للـ dataset وأعد التدريب.
3. **Teachable Machine:** ادخل على [teachablemachine.withgoogle.com](https://teachablemachine.withgoogle.com) ودرّب موديل بكاميرا اللابتوب على ٣ حاجات من حواليك. اكتب الفرق بين التجربتين.

<details><summary>حل مهمة ١</summary>

```python
mixed_X, mixed_y = [], []
for i, noise in enumerate([0.1, 0.3, 0.5]):
    Xi, yi = make_dataset(150, np.random.default_rng(10 + i), noise=noise)
    mixed_X.append(Xi)
    mixed_y.append(yi)
robust = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400, random_state=0)
robust.fit(np.vstack(mixed_X), np.concatenate(mixed_y))
for noise in [0.15, 0.45, 0.6]:
    Xn, yn = make_dataset(100, np.random.default_rng(1), noise=noise)
    print(f"noise={noise}: original={models['Neural Net'].score(Xn, yn):.0%}  robust={robust.score(Xn, yn):.0%}")
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## ✅ معايير التقييم
| المعيار | الدرجة |
|---|---|
| دربت وقارنت ٣ موديلات على الأقل | ٣٠٪ |
| حللت الأخطاء (confusion matrix) وشرحتها | ٢٠٪ |
| عملت تحسين واحد على الأقل وقست أثره بالأرقام | ٣٠٪ |
| عرض ٣ دقايق: المشكلة، اللي عملته، النتيجة | ٢٠٪ |

</div>